# KV Cache & Efficient Autoregressive Generation

In the last notebook, we saw how a decoder generates text **one token at a time** using:
- masked self-attention
- logits → softmax → sampling

In this notebook we answer:

> **How do LLMs generate long sequences efficiently without recomputing everything each time?**

Key idea: **KV cache** (key–value cache) for attention.

We'll keep this notebook short and visual, with minimal code for intuition.

## 1. The Problem: Naive Generation is Expensive

Suppose we want to generate a sequence of length **T** tokens.

In naive autoregressive generation, at each decoding step we:
- feed the **entire prefix** into the model
- recompute all attention operations from scratch

For step 1: sequence length = 1  → small
For step 2: sequence length = 2  → recompute attention for tokens 1–2
For step 3: sequence length = 3  → recompute attention for tokens 1–3
...
For step T: sequence length = T → recompute attention for tokens 1–T

Total cost grows roughly like:

```text
1 + 2 + 3 + ... + T  ≈  O(T²)
```

This is **too expensive** for long generations.

We need a way to **reuse past computation** instead of recomputing it at every step.

That is exactly what **KV cache** does.

## 2. Reminder: Q, K, V in Self-Attention

For each token, self-attention computes three vectors:

- **Q (Query)** – what this token is asking for
- **K (Key)** – how this token can be matched
- **V (Value)** – information carried by this token

At each layer, for all tokens, we have matrices:

- Q: `[seq_len, d_k]`
- K: `[seq_len, d_k]`
- V: `[seq_len, d_v]`

Attention uses:

```text
scores = Q @ Kᵀ
weights = softmax(scores)
output = weights @ V
```

Key observation:
> For a **new** token, we only need a **new query** Q_new, but the **keys and values of old tokens are unchanged.**

So instead of recomputing K and V for the whole sequence each time, we can **store them once** and reuse them.

This stored memory is called the **KV cache**.

## 3. What is KV Cache?

During generation, for each layer and each attention head, we maintain:

- `K_cache`: keys for all past tokens
- `V_cache`: values for all past tokens

At step `t`:
1. We compute **Q_t, K_t, V_t** only for the **new token**.
2. We **append** K_t and V_t to the caches:
   - `K_cache = [K_cache, K_t]`
   - `V_cache = [V_cache, V_t]`
3. We compute attention for the **new token only**:
   - `scores_t = Q_t @ K_cacheᵀ`
   - `output_t = softmax(scores_t) @ V_cache`

We **never recompute** K and V for previous tokens.

This changes the cost from roughly **O(T²)** to **O(T)** per generated sequence (per layer/head), which is a huge speedup in practice.

In short:

> **KV cache = storing past keys and values instead of recomputing them, so each step only does work for the new token.**

## 4. Tiny KV Cache Demo (Toy Example)

This is a **toy example** to show the **mechanics**, not a full Transformer.

- We'll pretend we have a single attention layer.
- We'll simulate the cache growing as we add tokens.
- We'll compute attention only for the newest token using cached K, V.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

d_model = 8   # model dimension (small for demo)
d_k = 8       # key/query dimension

# Random weight matrices for Q, K, V (toy)
W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

def project_qkv(x):
    """Project a single token embedding x into Q, K, V."""
    Q = x @ W_Q
    K = x @ W_K
    V = x @ W_V
    return Q, K, V

def attend_last_token(K_cache, V_cache, Q_t):
    """Compute attention output for the last token using cached K, V and new Q_t."""
    # K_cache, V_cache: [seq_len, d_k]
    # Q_t: [d_k]
    scores = (K_cache @ Q_t) / (d_k ** 0.5)   # [seq_len]
    weights = F.softmax(scores, dim=-1)       # [seq_len]
    output = weights @ V_cache                # [d_k]
    return output, weights

# Simulate embeddings for 4 tokens
seq_len = 4
embeddings = torch.randn(seq_len, d_model)

K_cache = []
V_cache = []

for t in range(seq_len):
    x_t = embeddings[t]  # embedding of token t
    Q_t, K_t, V_t = project_qkv(x_t)

    # Append to cache
    K_cache.append(K_t)
    V_cache.append(V_t)

    K_mat = torch.stack(K_cache, dim=0)  # [t+1, d_k]
    V_mat = torch.stack(V_cache, dim=0)  # [t+1, d_k]

    out_t, weights_t = attend_last_token(K_mat, V_mat, Q_t)

    print(f"Step {t+1} (sequence length = {t+1})")
    print("Attention weights for last token:", weights_t)
    print("Output vector shape:", out_t.shape)
    print("-")

Note what’s happening:

- At each step, the **cache grows by 1 token**.
- We only compute Q, K, V for the **new token**.
- Attention for the new token uses **all past K, V from the cache**.

This is the same idea scaled up to huge models and batches in real LLM inference.

## 5. KV Cache in Real Libraries (High-Level)

You don’t need to implement KV cache manually when using frameworks like:
- Hugging Face `transformers`
- vLLM
- many hosted APIs

They typically expose arguments like:

- `use_cache=True`
- `past_key_values`

The usual pattern is:

1. Call the model with the **full prompt**, get:
   - logits for next token
   - `past_key_values` (the initial KV cache)
2. For each next step, call the model with **only the new token** and the `past_key_values`.
3. The model updates and returns a new `past_key_values`.

So at high level, your generation loop looks like:

```text
prompt → model → logits, KV_cache
token_1 → model(KV_cache) → logits, KV_cache
token_2 → model(KV_cache) → logits, KV_cache
... and so on
```

## 6. Summary

In this short notebook we covered:

- The **inefficiency** of naive autoregressive generation
- How **Q, K, V** are used in attention
- The idea of **KV cache**:
  - store **keys and values** for past tokens
  - only compute Q, K, V for the **new token**
- How this makes long generations **much faster** in practice

Together with the previous notebook on **decoder & sampling**, you now have a complete picture of:
> **How LLMs generate text step-by-step, and how they do it efficiently.**